# 07 — Perluasan Besar: Arsip `angelfire.com` via Wayback Machine

Lanjutan dari [06_combined_data_v2_more_pbn.ipynb](06_combined_data_v2_more_pbn.ipynb)
(21.675 board). Diminta cari lagi data PBN karena terbukti efektif menaikkan
akurasi. Ditemukan link eksternal di `tistis.nl` menunjuk ke
`angelfire.com/games2/pbnarchive/pbn/` — situs aslinya sudah **mati**
(DNS `www.angelfire.com` tidak resolve, `angelfire.com` root menolak koneksi),
tapi seluruh isinya berhasil direcover lewat **Wayback Machine**
(`web.archive.org`, snapshot 2019-08-06): 57 arsip zip, 56 berhasil diunduh
utuh (`capgem00.zip` snapshotnya sendiri korup/truncated di Wayback, dilewati).

**Isi arsip**: 57 kejuaraan dunia 1996-2002 — Bermuda Bowl, Venice Cup,
Vanderbilt, World Bridge Team Olympiad, Cap Gemini, Cavendish Invitational,
Dutch Teams Final, European Team Championships, ACBL International Team
Trials, dan lainnya. 3 arsip (`etc99`, `eyc98`, `eyc00`) ternyata
**byte-identik** dengan file yang sudah dimiliki dari `tistis.nl` (sumber
sama, dua mirror berbeda) — dikeluarkan untuk mencegah duplikasi board.

**Verifikasi sebelum dipakai** (pola yang sama seperti sumber-sumber
sebelumnya): cek player-name tags untuk kontaminasi bot (GIB/WBridge5/dll.)
di ke-57 arsip — **bersih**, satu-satunya "hit" adalah nama manusia asli
("Jack Zhao") yang cocok pola regex `\bJACK\b` secara kebetulan.

**Bug crash serius ditemukan & diperbaiki**: 53 board dari arsip lama
(Dutch Teams Final 1996/1998, ETC 2001, WC98, Politiken 1997) punya kartu
duplikat/hilang akibat kesalahan transkripsi manual era 1990-2000an — tiap
tangan individual tetap terhitung 13 kartu, tapi total dek gabungan cuma
44-49 kartu unik (bukan 52). Memberi input ini ke `endplay`'s DDS solver
menyebabkan **segfault** (bukan exception biasa) — penyebab dua percobaan
komputasi DDS pertama gagal (deadlock multiprocessing, lalu crash langsung).
Diperbaiki dengan validasi 52-kartu-unik di
`src/features/dds.py::compute_dds_features()` sebelum memanggil `endplay`,
melindungi SEMUA sumber data (LIN maupun PBN) secara permanen.

In [1]:
import sys
import time
from pathlib import Path
from collections import Counter

import pandas as pd
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.parser import PBNParser
from src.preprocessing.dataset_builder import load_splits
from src.evaluation.metrics import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-15" / "outputs" / "angelfire_expansion"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Sumber PBN gabungan (1.390 file total)

In [2]:
pbn_boards = PBNParser().parse_directory(PROJECT_ROOT / "data" / "raw_pbn")
print(f"Total board dari seluruh sumber PBN (1.390 file): {len(pbn_boards)}")

def source_key(name):
    if name.startswith("angelfire_"):
        return "angelfire_" + name.split("_")[1]
    if name.startswith("tistis_"):
        return "tistis_" + name.split("_")[1]
    return name.rsplit("_", 1)[0]

event_cnt = Counter(source_key(b.source_file) for b in pbn_boards)
print(f"\n{len(event_cnt)} arsip berbeda. Contoh arsip angelfire.com baru:")
for k, v in sorted(event_cnt.items()):
    if k.startswith("angelfire_"):
        print(f"  {k:28s}: {v} board")

Total board dari seluruh sumber PBN (1.390 file): 47712

71 arsip berbeda. Contoh arsip angelfire.com baru:
  angelfire_bermud00          : 1840 board
  angelfire_bermud97          : 1357 board
  angelfire_birm2000          : 319 board
  angelfire_capgem01          : 1350 board
  angelfire_capgem02          : 1200 board
  angelfire_capgem97          : 1188 board
  angelfire_capgem98          : 1197 board
  angelfire_capgem99          : 1199 board
  angelfire_cavend00          : 297 board
  angelfire_cavend01          : 294 board
  angelfire_cavend97          : 118 board
  angelfire_cavend98          : 110 board
  angelfire_cavend99          : 192 board
  angelfire_china00           : 579 board
  angelfire_china98           : 256 board
  angelfire_dbosch01          : 427 board
  angelfire_dutch00           : 176 board
  angelfire_dutch01           : 176 board
  angelfire_dutch96           : 176 board
  angelfire_dutch97           : 175 board
  angelfire_dutch98           : 176 board
  a

## 2. Dataset gabungan v3

`data/processed_combined/` dibangun ulang dari `data/raw/` (606 file LIN) +
`data/raw_pbn/` (1.390 file PBN, termasuk 1.178 file baru angelfire.com) +
18 fitur DDS. Cache DDS (`data/combined_dds_cache_v3.csv`) dihitung ulang
untuk semua board baru secara sekuensial dengan checkpoint (37.489 board,
~2.9 jam) setelah percobaan paralel (`multiprocessing.Pool`) deadlock tanpa
sebab jelas di mesin ini — pelajaran: untuk komputasi panjang, keandalan
(checkpoint + single-process) lebih penting dari kecepatan.

In [3]:
df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed_combined")
X_train, y_train = df_train[feature_cols].values, df_train["label"].values
X_test, y_test = df_test[feature_cols].values, df_test["label"].values

print(f"Train: {X_train.shape}, Val: {df_val.shape}, Test: {X_test.shape}")
print(f"Jumlah kelas: {len(le.classes_)}")

D:\Bridge-Prediction\src\preprocessing\dataset_builder.py:296: DtypeWarning: Columns (0: _board_number) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(processed_dir / "train.csv")


Train: (34832, 182), Val: (7462, 192), Test: (7461, 182)
Jumlah kelas: 36


## 3. Latih ulang kandidat terbaik di data yang diperluas

In [4]:
XGB_ACC_TUNED = dict(
    n_estimators=300, max_depth=5, learning_rate=0.03,
    subsample=0.9, colsample_bytree=0.6, min_child_weight=5, reg_lambda=2.0,
    random_state=42, n_jobs=-1, eval_metric="mlogloss", verbosity=0,
)
LGBM_CLASS_WEIGHT = dict(
    n_estimators=300, max_depth=-1, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
    verbose=-1, class_weight="balanced",
)

results = []

t0 = time.time()
xgb_clf = XGBClassifier(**XGB_ACC_TUNED)
xgb_clf.fit(X_train, y_train)
proba = xgb_clf.predict_proba(X_test)
res = evaluate(y_test, proba.argmax(axis=1), proba, le, model_name="XGBoost acc-tuned (49.755 board)")
res["fit_seconds"] = round(time.time() - t0, 1)
results.append(res)
print_summary(res)

t0 = time.time()
lgbm_clf = LGBMClassifier(**LGBM_CLASS_WEIGHT)
lgbm_clf.fit(X_train, y_train)
proba2 = lgbm_clf.predict_proba(X_test)
res2 = evaluate(y_test, proba2.argmax(axis=1), proba2, le, model_name="LightGBM class_weight (49.755 board)")
res2["fit_seconds"] = round(time.time() - t0, 1)
results.append(res2)
print_summary(res2)


  XGBoost acc-tuned (49.755 board)
  Accuracy          : 0.5611
  Precision (macro) : 0.4615
  Recall (macro)    : 0.3145
  F1 (macro)        : 0.3416
  F1 (weighted)     : 0.5322
  Top-3 Accuracy    : 0.8175
  Top-5 Accuracy    : 0.8984


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM class_weight (49.755 board)
  Accuracy          : 0.5639
  Precision (macro) : 0.4543
  Recall (macro)    : 0.3915
  F1 (macro)        : 0.4101
  F1 (weighted)     : 0.5567
  Top-3 Accuracy    : 0.8208
  Top-5 Accuracy    : 0.8972


## 4. Bandingkan dengan baseline 21.675-board (sebelum perluasan angelfire.com)

In [5]:
comparison = compare_models(results)
comparison.to_csv(OUT_DIR / "test_comparison.csv")

# Angka referensi dari 06_combined_data_v2_more_pbn.ipynb (21.675 board)
reference = {
    "XGBoost acc-tuned (21.675 board, REF)": {"accuracy": 0.5405, "f1_macro": 0.2937, "f1_weighted": 0.5039},
    "LightGBM class_weight (21.675 board, REF)": {"accuracy": 0.5426, "f1_macro": 0.3487, "f1_weighted": 0.5185},
}
ref_df = pd.DataFrame(reference).T
print(ref_df)
print()
print(comparison[["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy"]])

                                           accuracy  f1_macro  f1_weighted
XGBoost acc-tuned (21.675 board, REF)        0.5405    0.2937       0.5039
LightGBM class_weight (21.675 board, REF)    0.5426    0.3487       0.5185

                                      accuracy  f1_macro  f1_weighted  \
model                                                                   
XGBoost acc-tuned (49.755 board)      0.561051  0.341633     0.532195   
LightGBM class_weight (49.755 board)  0.563865  0.410098     0.556675   

                                      top_3_accuracy  top_5_accuracy  
model                                                                 
XGBoost acc-tuned (49.755 board)            0.817451        0.898405  
LightGBM class_weight (49.755 board)        0.820802        0.897199  


In [6]:
import json

summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "data_sources": {
        "lin_files": 606,
        "pbn_files_total": 1390,
        "pbn_files_new_today": 1178,
        "pbn_source": "computerbridge.se (43) + tistis.nl (169) + angelfire.com via Wayback Machine (1178)",
        "pbn_boards_total": len(pbn_boards),
        "malformed_boards_excluded_from_dds": 53,
    },
    "dataset_size": {
        "total_boards": len(df_train) + len(df_val) + len(df_test),
        "train": len(df_train), "val": len(df_val), "test": len(df_test),
        "n_features": len(feature_cols),
        "n_classes": len(le.classes_),
    },
    "bugs_fixed": [
        "endplay DDS solver segfaults (uncatchable) on decks with duplicate/missing cards "
        "from old manual-transcription errors -- fixed with a 52-unique-card validation guard "
        "in compute_dds_features() before calling into the C library",
        "multiprocessing.Pool deadlocked with no clear cause on this machine for long-running "
        "DDS computation -- switched to sequential single-process execution with periodic "
        "CSV checkpointing so a crash never loses more than ~1000 boards of progress",
    ],
    "results": {r["model"]: {k: r[k] for k in ("accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy")} for r in results},
}
with open(OUT_DIR / "nb07_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "generated": "2026-07-16T23:02:22.048597",
  "data_sources": {
    "lin_files": 606,
    "pbn_files_total": 1390,
    "pbn_files_new_today": 1178,
    "pbn_source": "computerbridge.se (43) + tistis.nl (169) + angelfire.com via Wayback Machine (1178)",
    "pbn_boards_total": 47712,
    "malformed_boards_excluded_from_dds": 53
  },
  "dataset_size": {
    "total_boards": 49755,
    "train": 34832,
    "val": 7462,
    "test": 7461,
    "n_features": 182,
    "n_classes": 36
  },
  "bugs_fixed": [
    "endplay DDS solver segfaults (uncatchable) on decks with duplicate/missing cards from old manual-transcription errors -- fixed with a 52-unique-card validation guard in compute_dds_features() before calling into the C library",
    "multiprocessing.Pool deadlocked with no clear cause on this machine for long-running DDS computation -- switched to sequential single-process execution with periodic CSV checkpointing so a crash never loses more than ~1000 boards of progress"
  ],
  "result

## 5. Kesimpulan

**Hasil terbaik baru dari seluruh proyek, dan lompatan terbesar sejauh ini** —
1.178 file PBN tambahan dari arsip angelfire.com (21.675 → 49.755 board,
+130%) mengonfirmasi hipotesis: menambah data terbukti sangat efektif
menaikkan performa, terutama F1 macro (penanganan kelas langka):

| Model | Accuracy | F1 Macro | F1 Weighted |
|---|---|---|---|
| XGBoost acc-tuned (21.675 board) | 54.1% | 0.294 | 0.504 |
| **XGBoost acc-tuned (49.755 board)** | **56.1%** (+2.1pp) | **0.342** (+4.8pp) | **0.532** (+2.8pp) |
| LightGBM class_weight (21.675 board) | 54.3% | 0.349 | 0.519 |
| **LightGBM class_weight (49.755 board)** | **56.4%** (+2.1pp) | **0.410** (+6.1pp) | **0.557** (+3.8pp) |

**LightGBM+class_weight tetap kandidat terbaik proyek, unggul di SEMUA
metrik** dan margin keunggulan F1 macro atas XGBoost justru melebar
(0.410 vs 0.342, dari sebelumnya 0.349 vs 0.294) — konsisten dengan pola
yang diamati sejak 2026-07-09: lebih banyak data membantu class_weight
"melihat" kelas langka dengan lebih baik.

**Sumber data hampir habis untuk PBN non-BBO yang legitimate dan bersih
dari bot** — arsip angelfire.com ini kemungkinan adalah kumpulan terbesar
yang tersisa dari era pra-2010 (57 kejuaraan dunia). Perluasan berikutnya
kemungkinan perlu sumber jenis lain (bukan sekadar PBN archive tambahan)
jika ingin melanjutkan tren kenaikan ini.

**Rekomendasi**: `LightGBM+class_weight` di `data/processed_combined/`
(49.755 board, 182 fitur) adalah kandidat model utama proyek saat ini,
sudah dipromosikan ke pipeline resmi `notebooks_dds/`.